In [1]:
from utils import Sql, train_val_test_split, get_sm_service_role_arn
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
import pandas as pd
import sagemaker, boto3, utils
import paths as p

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [ ]:
# User vars
region = 'us-east-1'
model_package_group_name='abalone'
model_version=1
project_bucket='omm-test-bucket'
project_path = 'models/abalone'
target_name='rings'

In [ ]:
# Script vars
prediction_name=target_name+'_prediction'

data_dir_uri=f's3://{project_bucket}/{project_path}/data'
model_dir_uri=f's3://{project_bucket}/{project_path}/model'
train_dir = f'{data_dir_uri}/input/train'
validation_dir=f'{data_dir_uri}/input/validation'
test_X_file=f'{data_dir_uri}/input/test/text_X.csv'
test_y_file=f'{data_dir_uri}/input/test/text_y.csv'

batch_out_dir=f'{data_dir_uri}/batch-output'

baseline_X_file=f'{data_dir_uri}/baseline/baseline_X.csv'
baseline_file=f'{data_dir_uri}/baseline/baseline.csv'
baseline_model_out_file=f'{batch_out_dir}/baseline_X.csv.out'
baseline_pred_file=f'{data_dir_uri}/baseline/baseline_pred.csv'

role = get_sm_service_role_arn()
boto_session=boto3.Session(region_name=region)
sagemaker_session = sagemaker.Session(boto_session=boto_session)

In [ ]:
# DATA INGESTION
sql=Sql('user-1', 'password', 'db_1')

abalone_df = sql.query('SELECT * FROM abalone;')
abalone_df=abalone_df.drop(columns=['id','created_at'])

In [ ]:
# FEATURE ENGINEERING
abalone_df = pd.get_dummies(abalone_df, columns=['sex'], drop_first=False)
abalone_df[['sex_I', 'sex_M', 'sex_F']] = abalone_df[['sex_I', 'sex_M', 'sex_F']].astype(int)
abalone_df.hsdfs

,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings,sex_F,sex_I,sex_M
0,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15,0,0,1
1,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7,0,0,1
2,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9,1,0,0
3,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10,0,0,1
4,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7,0,1,0
5,0.425,0.300,0.095,0.3515,0.1410,0.0775,0.120,8,0,1,0
6,0.530,0.415,0.150,0.7775,0.2370,0.1415,0.330,20,1,0,0
7,0.545,0.425,0.125,0.7680,0.2940,0.1495,0.260,16,1,0,0
8,0.475,0.370,0.125,0.5095,0.2165,0.1125,0.165,9,0,0,1
9,0.550,0.440,0.150,0.8945,0.3145,0.1510,0.320,19,1,0,0


In [ ]:
# DATA FORMATTING
y = abalone_df['rings']
X = abalone_df.drop(columns=['rings'])

X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y, train_size=0.7, val_size=0.15, random_state=42)

pd.concat([y_train, X_train], axis=1).to_csv(f'{p.train_dir}/train.csv', index=False, header=False)
pd.concat([y_val, X_val], axis=1).to_csv(f'{p.validation_dir}/validation.csv', index=False, header=False)
pd.concat([y_val, X_val], axis=1).to_csv(p.baseline_file, index=False, header=True) # need header
X_test.to_csv(p.test_X_file, index=True, header=False)
y_test.to_csv(p.test_y_file, index=True, header=False)

input_data = {
    'train': sagemaker.inputs.TrainingInput(p.train_dir+'/', content_type='text/csv'),
    'validation': sagemaker.inputs.TrainingInput(p.validation_dir+'/', content_type='text/csv'),
    # xgboost only accepts train and validation 'test': sagemaker.inputs.TrainingInput(test_s3_dir+'/', content_type='text/csv')
}

In [ ]:
# TRAINING
estimator = sagemaker.estimator.Estimator(
    image_uri=sagemaker.image_uris.retrieve('xgboost', region, version='1.5-1'),
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=model_dir_uri,
    sagemaker_session=sagemaker_session,
    subnets=['subnet-001be661bcef4b615', 'subnet-003ad32933ca43e74'],
    security_group_ids=['sg-00f14515abe1e47e8']
)

# Set hyperparameters
estimator.set_hyperparameters(
    objective='reg:squarederror',
    num_round=100
)

# Train
estimator.fit(input_data)
print(estimator.model_data)

/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-05-21-17-07-18-342


2026-05-21 17:07:23 Starting - Starting the training job...
2026-05-21 17:07:41 Starting - Preparing the instances for training...
2026-05-21 17:08:00 Downloading - Downloading input data...
2026-05-21 17:08:26 Downloading - Downloading the training image...
2026-05-21 17:09:12 Training - Training image download completed. Training in progress../miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-05-21 17:09:22.455 ip-172-31-108-128.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-05-21 17:09:22.476 ip-172-31-108-128.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-05-21:17:09:22:INFO] Imported framework sagemaker_xgboost_container.training
[2026-05-21:17:09:22:INFO] Failed to parse hyperparameter objective value reg

In [ ]:
# BASELINING
# get baseline X
baseline=pd.read_csv(p.baseline_file, header=0)
baseline[target_name] = baseline[target_name].astype(float)
baseline_X = baseline.drop(columns=[target_name])
baseline_X.to_csv(p.baseline_X_file, index=False, header=False)

# Get predictions from baseline X
transformer = estimator.transformer(instance_count=1, instance_type='ml.m5.xlarge', output_path=p.batch_out_dir)
transformer.transform(data=p.baseline_X_file, content_type='text/csv', split_type='Line')
transformer.wait()

# Rename/move predictions file
utils.move_s3_file(boto_session, p.baseline_model_out_file, p.baseline_pred_file)

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-05-21-17-13-20-923
INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-05-21-17-13-21-446


............................../miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-05-21:17:18:27:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-21:17:18:27:INFO] No GPUs detected (normal if no gpus installed)
[2026-05-21:17:18:27:INFO] nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) {
      proxy_set_header X-Forwarded-For $proxy_add_

,rings_prediction,rings,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,sex_F,sex_I,sex_M
0,8.144435,7.0,0.410,0.30,0.100,0.2820,0.1255,0.057,0.0875,0,1,0
1,12.470012,13.0,0.450,0.36,0.125,0.4995,0.2035,0.100,0.1700,1,0,0
2,9.323292,11.0,0.490,0.38,0.135,0.5415,0.2175,0.095,0.1900,0,0,1
3,10.996564,14.0,0.430,0.34,0.125,0.3840,0.1375,0.061,0.1460,0,1,0
4,10.479134,11.0,0.545,0.45,0.150,0.8795,0.3870,0.150,0.2625,0,0,1


In [ ]:
# Ensure project folder looks like this

# - {project}/
# - data/
#   - train/
#     - train.csv
#   - validation/
#     - validation.csv
#   - test/
#     - test_X.csv
#     - test_y.csv
# - baseline/
#   - baseline.csv
#   - baseline_X.csv
#   - baseline_pred.csv

In [ ]:
# Register model from estimator
model = estimator.create_model(name=f"{model_package_group_name}-{str(model_version)}")

model.create(instance_type='ml.m5.large') # temporary instance for validation

model_package = model.register(
    content_types=['text/csv'],
    response_types=['text/csv'],
    inference_instances=['ml.m5.large'],
    transform_instances=['ml.m5.large'],
    model_package_group_name='abalone',
    description='XGBoost model for abalone ring prediction',
    approval_status='PendingManualApproval'  # or 'Approved'
)

INFO:sagemaker:Creating model with name: abalone-1
